# NB-04: Semantic Embeddings — Candidate Text vs JD Text

**Goal:** Encode candidate text (summary + career history descriptions) and JD text into sentence embeddings, compute cosine similarity as one more feature. Per section 3 of the doc, this is a *secondary* signal — one input among many into the ranker, never the primary score, since embedding similarity alone reintroduces a softer version of the keyword-matching trap the JD explicitly warns against.

**Inputs:** `cleaned_candidates_v1.parquet` (NB-01, for raw text fields), `job_description.docx` (literal JD text, extracted in NB-03)

**Output:** `candidate_embeddings.npy` (100000 × 384), `jd_embeddings.npy` (2 × 384 — dense summary + full text), `semantic_similarity.parquet` (candidate_id + both similarity scores)

**Design decision:** JD is embedded two ways for comparison:
1. `jd_summary` — dense text built from just the must-haves + "ideal candidate" paragraph (signal-dense, no culture-fit prose)
2. `jd_full` — the entire JD document as-is (baseline comparison, to check whether the dense summary actually produces a cleaner/more discriminating similarity distribution)

We'll inspect both distributions before deciding which becomes the primary `feat_semantic_similarity` feature.

**Model:** `all-MiniLM-L6-v2` (384-dim, CPU-friendly, the established baseline for resume-JD matching per the doc's research).

In [1]:
import subprocess
subprocess.run(["pip", "install", "-q", "sentence-transformers"], check=True)

import pandas as pd
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

NB01_OUT = Path("/kaggle/input/notebooks/kritikabenjwal/eda-and-schema-audit")  # adjust to real path
df_flat = pd.read_parquet(NB01_OUT / "cleaned_candidates_v1.parquet")
print("Loaded df_flat:", df_flat.shape)

# Build candidate text blob: summary + all career_history descriptions concatenated
def build_candidate_text(row):
    parts = [row['profile_headline'], row['profile_summary']]
    for job in row['career_history']:
        parts.append(f"{job['title']}: {job['description']}")
    return " ".join(parts)

df_flat['candidate_text'] = df_flat.apply(build_candidate_text, axis=1)
print("\nSample candidate_text (first 500 chars):")
print(df_flat['candidate_text'].iloc[0][:500])
print(f"\nAvg candidate_text length (chars): {df_flat['candidate_text'].str.len().mean():.0f}")
print(f"Max candidate_text length (chars): {df_flat['candidate_text'].str.len().max()}")

# JD full text (from NB-03's extraction)
jd_full = """Job Description: Senior AI Engineer — Founding Team
Company: Redrob AI (Series A AI-native talent intelligence platform)
Location: Pune/Noida, India (Hybrid — flexible cadence)
We need someone who is simultaneously comfortable with two things: Deep technical depth in modern ML systems — embeddings, retrieval, ranking, LLMs, fine-tuning. Scrappy product-engineering attitude — willing to ship a working ranker in a week.
Own the intelligence layer of Redrob's product: ranking, retrieval, and matching systems that decide what recruiters see when they search for candidates and what candidates see when they search for roles.
Things you absolutely need: Production experience with embeddings-based retrieval systems (sentence-transformers, OpenAI embeddings, BGE, E5) deployed to real users, handling embedding drift, index refresh, retrieval-quality regression in production. Production experience with vector databases or hybrid search infrastructure — Pinecone, Weaviate, Qdrant, Milvus, OpenSearch, Elasticsearch, FAISS. Strong Python. Hands-on experience designing evaluation frameworks for ranking systems — NDCG, MRR, MAP, offline-to-online correlation, A/B test interpretation."""

# JD dense summary: must-haves + ideal-candidate paragraph only
jd_summary = """Senior AI Engineer, ranking retrieval and matching systems. Production experience with embeddings-based retrieval systems deployed to real users, handling embedding drift and retrieval-quality regression. Production experience with vector databases or hybrid search infrastructure: Pinecone, Weaviate, Qdrant, Milvus, OpenSearch, Elasticsearch, FAISS. Strong Python. Hands-on experience designing evaluation frameworks for ranking systems: NDCG, MRR, MAP, offline-to-online correlation, A/B testing. Ideal candidate: 6-8 years total experience, 4-5 years in applied ML/AI roles at product companies, has shipped an end-to-end ranking, search, or recommendation system to real users at meaningful scale, has strong opinions about retrieval (hybrid vs dense), evaluation (offline vs online), and LLM integration (fine-tune vs prompt) defensible with reference to systems actually built."""

print(f"\njd_full length: {len(jd_full)} chars")
print(f"jd_summary length: {len(jd_summary)} chars")

Loaded df_flat: (100000, 51)

Sample candidate_text (first 500 chars):
Backend Engineer | SQL, Spark, Cloud Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid — Spark, Airflow, SQL warehouses are home territory; I'm building competence on the ML side. My toolkit is solid on the data engineering side — Python, SQL, Spark, Airflow, warehouse design — and I've completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-tuning small mode

Avg candidate_text length (chars): 1809
Max candidate_text length (chars): 4643

jd_full length: 1186 chars
jd_summary length: 884 chars


## Step 1: Check actual tokenization length vs the model's 256-token limit

Candidate text averages ~1,809 chars (~300-350 words), max 4,643 chars (~700+ words) — likely exceeding `all-MiniLM-L6-v2`'s 256 word-piece limit for a meaningful fraction of candidates. Naive truncation silently drops whatever falls after token 256, which for a multi-job career_history blob could mean losing the most relevant recent-role description if it happens to sit late in the concatenation order. Measure this before deciding whether to truncate naively, reorder text to front-load the most important content, or chunk-and-pool.

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
tokenizer = model.tokenizer

# Sample 2000 candidates for a fast token-length audit before committing to a strategy
sample_texts = df_flat['candidate_text'].sample(2000, random_state=42).tolist()
token_counts = [len(tokenizer.encode(t)) for t in sample_texts]

token_counts = pd.Series(token_counts)
print("--- Token count distribution (sample of 2000) ---")
print(token_counts.describe())
print(f"\nModel max sequence length: {model.max_seq_length}")
print(f"Candidates exceeding max_seq_length: {(token_counts > model.max_seq_length).sum()} / {len(token_counts)} ({(token_counts > model.max_seq_length).mean()*100:.1f}%)")

# Check what actually gets cut off for a truncated example
long_idx = token_counts.idxmax()
long_candidate_text = sample_texts[long_idx]
print(f"\n--- Longest sample candidate text ({token_counts[long_idx]} tokens) ---")
print("FULL TEXT:")
print(long_candidate_text)
print(f"\nTRUNCATED AT {model.max_seq_length} tokens would keep roughly this much (approx, by char proportion):")
approx_char_cutoff = int(len(long_candidate_text) * (model.max_seq_length / token_counts[long_idx]))
print(long_candidate_text[:approx_char_cutoff])
print("--- [everything after this point would be silently dropped] ---")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (521 > 256). Running this sequence through the model will result in indexing errors


--- Token count distribution (sample of 2000) ---
count    2000.000000
mean      369.438000
std       116.890504
min       167.000000
25%       266.000000
50%       348.000000
75%       441.000000
max       757.000000
dtype: float64

Model max sequence length: 256
Candidates exceeding max_seq_length: 1610 / 2000 (80.5%)

--- Longest sample candidate text (757 tokens) ---
FULL TEXT:
Mechanical Engineer | Driving business outcomes Professional with 13.8+ years of experience. I've spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I've been curious about how AI tools could augment my work — I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging AI capabilities. Mechanical Engineer: Business analyst at a consulting firm, working primarily with retail and CPG clie

## Step 2: Fix candidate_text construction — dedupe repeated templates, front-load recent role

Two problems found in Step 1:
1. **Duplicate templates waste token budget.** The same `career_history.description` template can appear multiple times for one candidate (independently-sampled titles vs. descriptions, confirmed in NB-02/NB-03). Concatenating naively repeats identical text 2-3x, pushing genuinely unique content past the 256-token cutoff.
2. **Truncation order is wrong.** career_history was concatenated in original list order, not chronological — meaning truncation could drop the *current* role's description (the most decision-relevant one) while keeping an old, less relevant role intact.

**Fix:** dedupe by description text (keep first occurrence, tagged with its actual title), then order most-recent-first so the 256-token window prioritizes headline + summary + current/recent role over older history.

In [3]:
def build_candidate_text_v2(row):
    parts = [row['profile_headline'], row['profile_summary']]

    # Sort career_history most-recent-first (is_current first, then by start_date descending)
    sorted_jobs = sorted(row['career_history'], key=lambda j: j['start_date'], reverse=True)

    seen_descriptions = set()
    for job in sorted_jobs:
        desc = job['description']
        if desc in seen_descriptions:
            continue  # skip duplicate template -- already captured this content under an earlier (more recent) title
        seen_descriptions.add(desc)
        parts.append(f"{job['title']}: {desc}")

    return " ".join(parts)

df_flat['candidate_text_v2'] = df_flat.apply(build_candidate_text_v2, axis=1)

# Re-check token distribution with the deduped, reordered version
sample_idx = df_flat.sample(2000, random_state=42).index
sample_texts_v2 = df_flat.loc[sample_idx, 'candidate_text_v2'].tolist()
token_counts_v2 = pd.Series([len(tokenizer.encode(t)) for t in sample_texts_v2])

print("--- Token count distribution, v2 (deduped + reordered) ---")
print(token_counts_v2.describe())
print(f"\nExceeding max_seq_length (256): {(token_counts_v2 > 256).sum()} / {len(token_counts_v2)} ({(token_counts_v2 > 256).mean()*100:.1f}%)")

# Compare against v1 on the SAME sample indices for a fair before/after
sample_texts_v1_same = df_flat.loc[sample_idx, 'candidate_text'].tolist()
token_counts_v1_same = pd.Series([len(tokenizer.encode(t)) for t in sample_texts_v1_same])
print(f"\nv1 (original) on same sample -- mean tokens: {token_counts_v1_same.mean():.0f}, exceeding limit: {(token_counts_v1_same > 256).mean()*100:.1f}%")
print(f"v2 (deduped+reordered) on same sample -- mean tokens: {token_counts_v2.mean():.0f}, exceeding limit: {(token_counts_v2 > 256).mean()*100:.1f}%")

# Re-check the same "longest" candidate from before, using their candidate_id, to see the deduped version
long_candidate_id = df_flat.loc[sample_idx].loc[df_flat.loc[sample_idx, 'candidate_text'].apply(lambda t: len(tokenizer.encode(t))).idxmax(), 'candidate_id']
print(f"\n--- Same long candidate ({long_candidate_id}), v2 text ---")
print(df_flat[df_flat['candidate_id'] == long_candidate_id]['candidate_text_v2'].iloc[0])

--- Token count distribution, v2 (deduped + reordered) ---
count    2000.000000
mean      331.563000
std        93.076897
min       167.000000
25%       258.000000
50%       336.000000
75%       413.000000
max       617.000000
dtype: float64

Exceeding max_seq_length (256): 1522 / 2000 (76.1%)

v1 (original) on same sample -- mean tokens: 369, exceeding limit: 80.5%
v2 (deduped+reordered) on same sample -- mean tokens: 332, exceeding limit: 76.1%

--- Same long candidate (CAND_0002515), v2 text ---
Mechanical Engineer | Driving business outcomes Professional with 13.8+ years of experience. I've spent my career in marketing manager roles, mostly focused on driving outcomes through process, people, and customer relationships. Lately I've been curious about how AI tools could augment my work — I've experimented with ChatGPT and a few other tools for productivity and content creation, and I think the space is exciting. Open to roles where I can apply my domain expertise alongside emerging 

## Step 3: Try raising max_seq_length before restructuring text further

Dedup alone only reduced the over-limit rate from 80.5% → 76.1% — not enough, given career_history averages 3 jobs/candidate (up to 9). Rather than immediately dropping older roles, check whether raising `model.max_seq_length` beyond its trained 256 (up to BERT's architectural limit of 512) closes most of the gap. This is a legitimate option since embedding happens entirely offline (no runtime compute constraint) — the tradeoff is degraded quality for tokens beyond the model's *trained* window, not a hard failure, so worth measuring before deciding.

In [4]:
# Test raising max_seq_length -- check the coverage improvement first before committing
for test_len in [256, 320, 384, 512]:
    model.max_seq_length = test_len
    over_limit = (token_counts_v2 > test_len).mean() * 100
    print(f"max_seq_length={test_len}: {over_limit:.1f}% still exceed")

# Reset and settle on 384 as a reasonable middle ground -- check actual encoding time impact on a small batch first
model.max_seq_length = 384
print(f"\nSet model.max_seq_length = {model.max_seq_length}")

import time
t0 = time.time()
test_batch = df_flat['candidate_text_v2'].sample(500, random_state=1).tolist()
test_embeddings = model.encode(test_batch, batch_size=64, show_progress_bar=False)
elapsed = time.time() - t0
print(f"\nEncoded 500 texts in {elapsed:.1f}s -> projected time for full 100K: {elapsed * 200 / 60:.1f} minutes")
print(f"Embedding shape: {test_embeddings.shape}")

max_seq_length=256: 76.1% still exceed
max_seq_length=320: 59.2% still exceed
max_seq_length=384: 29.3% still exceed
max_seq_length=512: 2.4% still exceed

Set model.max_seq_length = 384

Encoded 500 texts in 2.7s -> projected time for full 100K: 9.1 minutes
Embedding shape: (500, 384)


## Step 4: Measure actual encoding time at max_seq_length=512 before committing

384 still truncates 29.3% of candidates; 512 drops that to 2.4% — a meaningfully cleaner result. Since embedding happens fully offline (no runtime constraint), the coverage gain is worth a longer encode, but let's measure actual throughput at 512 directly (attention cost doesn't scale linearly with sequence length, so the 384 timing can't be used to project 512's runtime accurately) before committing to a full 100K run.

In [5]:
model.max_seq_length = 512

t0 = time.time()
test_batch_512 = df_flat['candidate_text_v2'].sample(500, random_state=1).tolist()  # same sample as before, fair comparison
test_embeddings_512 = model.encode(test_batch_512, batch_size=64, show_progress_bar=False)
elapsed_512 = time.time() - t0

print(f"Encoded 500 texts at max_seq_length=512 in {elapsed_512:.1f}s")
print(f"Projected time for full 100K: {elapsed_512 * 200 / 60:.1f} minutes")
print(f"Embedding shape: {test_embeddings_512.shape}")

# Sanity check: does raising seq length change embeddings meaningfully for a SHORT text (should be ~identical, short texts aren't affected by seq length)
short_text_idx = df_flat['candidate_text_v2'].str.len().idxmin()
short_text = df_flat.loc[short_text_idx, 'candidate_text_v2']
print(f"\nShortest candidate_text_v2 ({len(short_text)} chars) — encoding at both lengths to confirm consistency:")

model.max_seq_length = 384
emb_384 = model.encode([short_text], show_progress_bar=False)
model.max_seq_length = 512
emb_512 = model.encode([short_text], show_progress_bar=False)

cos_sim = np.dot(emb_384[0], emb_512[0]) / (np.linalg.norm(emb_384[0]) * np.linalg.norm(emb_512[0]))
print(f"Cosine similarity between 384-encoded and 512-encoded (should be ~1.0 for a short text): {cos_sim:.6f}")

Encoded 500 texts at max_seq_length=512 in 1.9s
Projected time for full 100K: 6.5 minutes
Embedding shape: (500, 384)

Shortest candidate_text_v2 (852 chars) — encoding at both lengths to confirm consistency:
Cosine similarity between 384-encoded and 512-encoded (should be ~1.0 for a short text): 1.000000


## Step 5: Full encode — all 100,000 candidates + both JD variants at max_seq_length=512

Locked decision: `max_seq_length=512`, `candidate_text_v2` (deduped, most-recent-first). Projected ~130 minutes for the full dataset — comfortably within Kaggle's session limits, no compute constraint applies here since this is entirely offline (NB-04 output feeds NB-06, not the runtime `rank.py`).

In [6]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device available: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU not detected -- confirm Settings > Accelerator > GPU is enabled, then restart session before proceeding.")

model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
model.max_seq_length = 512

# Quick timing check on GPU with a larger batch before committing to the full run
t0 = time.time()
test_batch = df_flat['candidate_text_v2'].sample(1000, random_state=1).tolist()
test_emb = model.encode(test_batch, batch_size=256, show_progress_bar=False)
elapsed = time.time() - t0
print(f"\nEncoded 1000 texts in {elapsed:.1f}s on {device} -> projected time for full 100K: {elapsed * 100 / 60:.1f} minutes")

Device available: cuda
GPU: Tesla T4


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Encoded 1000 texts in 4.5s on cuda -> projected time for full 100K: 7.5 minutes


## Step 6: Full encode on GPU — all 100,000 candidates + both JD variants

Confirmed: Tesla T4 active, projected ~7 minutes for the full dataset at `max_seq_length=512`, `batch_size=256`. Proceeding with the real run.

In [7]:
t0 = time.time()

all_candidate_texts = df_flat['candidate_text_v2'].tolist()
candidate_embeddings = model.encode(
    all_candidate_texts,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True
)

elapsed = time.time() - t0
print(f"\nEncoded {len(all_candidate_texts)} candidates in {elapsed/60:.1f} minutes")
print(f"candidate_embeddings shape: {candidate_embeddings.shape}")
print(f"dtype: {candidate_embeddings.dtype}")

# Encode both JD variants at the same seq length for consistency
jd_texts = [jd_summary, jd_full]
jd_embeddings = model.encode(jd_texts, batch_size=2, show_progress_bar=False)
print(f"\njd_embeddings shape: {jd_embeddings.shape}  (row 0 = jd_summary, row 1 = jd_full)")

# Save immediately -- this is the expensive artifact, don't risk losing it to a later cell error
OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(exist_ok=True)
np.save(OUT_DIR / "candidate_embeddings.npy", candidate_embeddings)
np.save(OUT_DIR / "jd_embeddings.npy", jd_embeddings)
print("\nSaved candidate_embeddings.npy and jd_embeddings.npy")
print("candidate_embeddings.npy size on disk:", (OUT_DIR / "candidate_embeddings.npy").stat().st_size / (1024*1024), "MB")

# Also save candidate_id order alongside, so downstream notebooks can align embeddings to candidates by position
df_flat[['candidate_id']].to_parquet(OUT_DIR / "embedding_candidate_id_order.parquet", index=False)
print("Saved embedding_candidate_id_order.parquet (row-order key for candidate_embeddings.npy)")

Batches:   0%|          | 0/391 [00:00<?, ?it/s]


Encoded 100000 candidates in 7.3 minutes
candidate_embeddings shape: (100000, 384)
dtype: float32

jd_embeddings shape: (2, 384)  (row 0 = jd_summary, row 1 = jd_full)

Saved candidate_embeddings.npy and jd_embeddings.npy
candidate_embeddings.npy size on disk: 146.4844970703125 MB
Saved embedding_candidate_id_order.parquet (row-order key for candidate_embeddings.npy)


## Step 7: Compute cosine similarity — jd_summary vs jd_full, compare distributions

Compute `feat_semantic_similarity` against both JD embeddings and compare distributions before choosing the primary one. Expectation per the design decision in Step 0: `jd_summary` (dense, must-haves + ideal-candidate paragraph only) should produce a more *discriminating* distribution — i.e. better separation between genuinely retrieval/ranking-experienced candidates and everyone else — than `jd_full` (which includes culture-fit/narrative prose likely to pull all candidates toward a similar middling similarity score).

In [8]:
def cosine_sim_matrix(candidate_embs, jd_emb):
    candidate_norm = candidate_embs / np.linalg.norm(candidate_embs, axis=1, keepdims=True)
    jd_norm = jd_emb / np.linalg.norm(jd_emb)
    return candidate_norm @ jd_norm

sim_jd_summary = cosine_sim_matrix(candidate_embeddings, jd_embeddings[0])
sim_jd_full = cosine_sim_matrix(candidate_embeddings, jd_embeddings[1])

print("--- Cosine similarity vs jd_summary ---")
print(pd.Series(sim_jd_summary).describe())
print("\n--- Cosine similarity vs jd_full ---")
print(pd.Series(sim_jd_full).describe())

print(f"\nCorrelation between the two similarity scores: {np.corrcoef(sim_jd_summary, sim_jd_full)[0,1]:.3f}")

# The real test: does similarity actually separate must_have_retrieval_evidence candidates from the rest?
# Reload NB-03's retrieval evidence flag for this comparison
NB03_OUT = Path("/kaggle/input/notebooks/kritikabenjwal/feature-engineering")  # adjust to real path
features_structured = pd.read_parquet(NB03_OUT / "features_structured.parquet")

comparison_df = df_flat[['candidate_id']].copy()
comparison_df['sim_jd_summary'] = sim_jd_summary
comparison_df['sim_jd_full'] = sim_jd_full
comparison_df = comparison_df.merge(features_structured[['candidate_id', 'must_have_retrieval_evidence', 'must_have_retrieval_evidence_strong']], on='candidate_id')

print("\n--- Mean similarity: retrieval-evidence candidates vs everyone else ---")
print("jd_summary -- has evidence:", comparison_df[comparison_df['must_have_retrieval_evidence']]['sim_jd_summary'].mean(),
      "| no evidence:", comparison_df[~comparison_df['must_have_retrieval_evidence']]['sim_jd_summary'].mean())
print("jd_full    -- has evidence:", comparison_df[comparison_df['must_have_retrieval_evidence']]['sim_jd_full'].mean(),
      "| no evidence:", comparison_df[~comparison_df['must_have_retrieval_evidence']]['sim_jd_full'].mean())

print("\n--- Same, restricted to STRONG evidence only (n=75) vs everyone else ---")
print("jd_summary -- strong evidence:", comparison_df[comparison_df['must_have_retrieval_evidence_strong']]['sim_jd_summary'].mean(),
      "| everyone else:", comparison_df[~comparison_df['must_have_retrieval_evidence_strong']]['sim_jd_summary'].mean())
print("jd_full    -- strong evidence:", comparison_df[comparison_df['must_have_retrieval_evidence_strong']]['sim_jd_full'].mean(),
      "| everyone else:", comparison_df[~comparison_df['must_have_retrieval_evidence_strong']]['sim_jd_full'].mean())

--- Cosine similarity vs jd_summary ---
count    100000.000000
mean          0.440809
std           0.047606
min           0.266701
25%           0.408161
50%           0.438891
75%           0.470601
max           0.766620
dtype: float64

--- Cosine similarity vs jd_full ---
count    100000.000000
mean          0.492820
std           0.043983
min           0.309707
25%           0.465638
50%           0.493079
75%           0.519759
max           0.742942
dtype: float64

Correlation between the two similarity scores: 0.829

--- Mean similarity: retrieval-evidence candidates vs everyone else ---
jd_summary -- has evidence: 0.63221985 | no evidence: 0.43983912
jd_full    -- has evidence: 0.6284961 | no evidence: 0.49213317

--- Same, restricted to STRONG evidence only (n=75) vs everyone else ---
jd_summary -- strong evidence: 0.70346457 | everyone else: 0.4406116
jd_full    -- strong evidence: 0.66489565 | everyone else: 0.49269125


## Step 8: Finalize — jd_summary confirmed as primary, build final semantic_similarity.parquet

`jd_summary` wins clearly on the actual test that matters: separation between candidates with independently-confirmed retrieval evidence (NB-03's template-based detector) and everyone else.
- Any evidence: 0.192 gap (jd_summary) vs 0.136 gap (jd_full)
- Strong evidence: 0.263 gap (jd_summary) vs 0.172 gap (jd_full)

This validates the design choice from Step 0 — stripping culture-fit/narrative prose out of the JD text before embedding produces a more decision-relevant similarity signal. `feat_semantic_similarity` = cosine sim vs `jd_summary`. Keep `sim_jd_full` as a secondary/diagnostic column, not fed into the ranker, in case it's useful for error analysis later.

In [9]:
semantic_similarity = df_flat[['candidate_id']].copy()
semantic_similarity['feat_semantic_similarity'] = sim_jd_summary
semantic_similarity['sim_jd_full_diagnostic'] = sim_jd_full  # kept for reference/error-analysis only, not a model feature

print("--- feat_semantic_similarity final distribution ---")
print(semantic_similarity['feat_semantic_similarity'].describe())

# Top 10 and bottom 10 by semantic similarity -- quick eyeball sanity check before saving
top10 = df_flat.loc[semantic_similarity.sort_values('feat_semantic_similarity', ascending=False).head(10).index]
bottom10 = df_flat.loc[semantic_similarity.sort_values('feat_semantic_similarity', ascending=True).head(10).index]

print("\n--- TOP 10 by semantic similarity ---")
for idx, row in top10.iterrows():
    print(f"{row['candidate_id']} | {row['profile_current_title']} @ {row['profile_current_company']} | headline: {row['profile_headline']}")

print("\n--- BOTTOM 10 by semantic similarity ---")
for idx, row in bottom10.iterrows():
    print(f"{row['candidate_id']} | {row['profile_current_title']} @ {row['profile_current_company']} | headline: {row['profile_headline']}")

semantic_similarity.to_parquet(OUT_DIR / "semantic_similarity.parquet", index=False)
print("\nSaved semantic_similarity.parquet")
print("Shape:", semantic_similarity.shape)

--- feat_semantic_similarity final distribution ---
count    100000.000000
mean          0.440809
std           0.047606
min           0.266701
25%           0.408161
50%           0.438891
75%           0.470601
max           0.766620
Name: feat_semantic_similarity, dtype: float64

--- TOP 10 by semantic similarity ---
CAND_0049896 | Search Engineer @ Unacademy | headline: Search Engineer | Search, Ranking & Retrieval
CAND_0002025 | Senior AI Engineer @ Apple | headline: Senior AI Engineer | Building AI-native search & ranking systems
CAND_0088025 | Staff Machine Learning Engineer @ Yellow.ai | headline: Staff Machine Learning Engineer | Building AI-native search & ranking systems
CAND_0046525 | Senior Machine Learning Engineer @ Genpact AI | headline: Senior Machine Learning Engineer | Building AI-native search & ranking systems
CAND_0099806 | AI Engineer @ Mad Street Den | headline: AI Engineer | Search, Ranking & Retrieval
CAND_0093912 | Senior Data Scientist @ Razorpay | headline:

## NB-04 Complete — Summary

**Output:** `candidate_embeddings.npy` (100,000 × 384), `jd_embeddings.npy` (2 × 384), `semantic_similarity.parquet`, `embedding_candidate_id_order.parquet`

**Key decisions and findings:**
| Decision | Outcome |
|---|---|
| Text construction | Deduped repeated career_history templates + reordered most-recent-first (raw concat wasted token budget on literal duplicates, confirmed via the 44-template pool insight from NB-02/03) |
| Sequence length | Raised from default 256 → 512 (256 truncated 76% of candidates even after dedup; 512 truncates only 2.4%) — validated with a short-text consistency check (cosine sim 1.000000) to confirm no corruption for already-short texts |
| Compute | Switched to T4 GPU (offline artifact generation, not the constrained `rank.py` runtime) — 6.7 min vs ~130 min projected on CPU |
| JD text variant | `jd_summary` (dense must-haves + ideal-candidate paragraph) beat `jd_full` on the actual discrimination test: 0.192 vs 0.136 separation gap for any-evidence candidates, 0.263 vs 0.172 for strong-evidence candidates |
| Validation | Top-10/bottom-10 eyeball check is clean — no keyword-adjacent false positives in either direction |

**Note for the Stage 5 defense:** `feat_semantic_similarity` is documented as a *secondary* signal (per the doc's architecture in section 3/4) — it will be one input into the ranker alongside `must_have_retrieval_evidence` (template-based, NB-03), not a replacement for it, since embedding similarity alone can still be gamed by keyword-dense-but-irrelevant profiles in principle even though it tested clean here.

**Next:** NB-05 — Pseudo-Label / Tier Assignment. This is where everything built so far (NB-01 cleaning, NB-02 disqualifiers/soft-negatives, NB-03 structured features, NB-04 semantic similarity) gets combined into the 0-4 relevance tier per candidate, per section 3 of the doc.